# Silver Load — Inventory Transactions Notebook
**ILLUSTRATIVE EXAMPLE — artificial lakehouse, schema, and table names.**

Mirrors the pattern used in the capstone project: a Fabric notebook (PySpark / Spark SQL) that stages bronze D365 F&O data landed by Azure Synapse Link, applies business logic, and materializes curated silver Delta tables in the analytics lakehouse. The notebook is executed daily by a scheduled Fabric Data Pipeline; the table list is passed in as a pipeline parameter.

In [ ]:
# Parameters cell — populated by the pipeline (ForEach over tableList)
table_list = [
    {"source": "legacy_sqlserver.dbo.PackingSlipLines", "target": "silver.PackingSlip_Fact"},
    {"source": "legacy_sqlserver.dbo.ItemMaster",       "target": "silver.Product_Dim"}
]

In [ ]:
%%pyspark
# Stage inventory-related journal entries from bronze (Synapse Link) tables
stage_df = spark.sql("""
    SELECT
        ma.MAINACCOUNTID,
        gje.SUBLEDGERVOUCHER,
        CAST(gje.ACCOUNTINGDATE AS DATE)    AS TransactionDate,
        SUM(gjae.TRANSACTIONCURRENCYAMOUNT) AS TransactionAmount
    FROM mfg_prod.finop.GeneralJournalEntry gje
    JOIN mfg_prod.finop.GeneralJournalAccountEntry gjae
         ON gjae.GENERALJOURNALENTRY = gje.RECID
    JOIN mfg_prod.finop.DimensionAttributeValueCombination davc
         ON gjae.LEDGERDIMENSION = davc.RECID
    JOIN mfg_prod.finop.MainAccount ma
         ON davc.MAINACCOUNT = ma.RECID
    WHERE gjae.POSTINGTYPE IN (93, 94)
    GROUP BY ma.MAINACCOUNTID, gje.SUBLEDGERVOUCHER, CAST(gje.ACCOUNTINGDATE AS DATE)
""")

stage_df.createOrReplaceTempView("Stage")
print(f"Staged rows: {stage_df.count():,}")

In [ ]:
%%sql
-- Rebuild curated silver fact from the staged view
DROP TABLE IF EXISTS MFG_ANALYTICS.silver.InventoryJournalTransaction_Fact;

CREATE TABLE MFG_ANALYTICS.silver.InventoryJournalTransaction_Fact
USING DELTA AS
SELECT *, current_timestamp() AS LoadTimestamp
FROM Stage;

In [ ]:
# Copy legacy SQL Server tables listed in the pipeline parameter into silver
for t in table_list:
    src, tgt = t["source"], t["target"]
    df = spark.read.table(src)          # staged by the pipeline Copy activity
    (df.dropDuplicates()
       .write.mode("overwrite")
       .format("delta")
       .saveAsTable(f"MFG_ANALYTICS.{tgt}"))
    print(f"Loaded {src} -> MFG_ANALYTICS.{tgt}: {df.count():,} rows")

In [ ]:
%%sql
-- Post-load validation: duplicate grain check must return 0
SELECT COUNT(*) AS DuplicateGrainRows FROM (
    SELECT MAINACCOUNTID, SUBLEDGERVOUCHER, TransactionDate
    FROM MFG_ANALYTICS.silver.InventoryJournalTransaction_Fact
    GROUP BY MAINACCOUNTID, SUBLEDGERVOUCHER, TransactionDate
    HAVING COUNT(*) > 1
)

**Scheduling:** this notebook is invoked by a Fabric Data Pipeline (ForEach → Copy Data → Notebook) that runs every day at 6:00 AM Eastern, with e-mail failure notification to the pipeline owner. Downstream Power BI reports pick up the refreshed silver/gold tables through the semantic model.